# ProVe-Arabic — Segmenter Benchmark

The empirical comparison of pysbd, Stanza, and spaCy for Arabic sentence-boundary detection that
motivates the choice in Section 3.3.4 and is reported in Section 5.3.1. Referenced by Appendix A.10.

**Not on the pipeline run path.** This notebook installs Stanza and spaCy, which are not pipeline
dependencies, and downloads a 459 MB Stanza model.

**Requirements:** Colab with a **T4 GPU**. The model and datasets download automatically on first run. **Run cell 1, restart the runtime when prompted, then continue.**

**This notebook is self-contained**: the setup cells below define every function it uses, so it
can be run on its own and in any order relative to the other notebooks.

**Expected findings:** pysbd is the most consistent across document types; Stanza collapses on
every document (2–61 segments, medians 899–6,999 characters); spaCy is unstable, matching pysbd on
prose but returning 27 segments for a 350,361-character list-heavy page.

## 1 — Setup  *(restart the runtime after the install cell)*

The Stage B module is reproduced here so this notebook runs standalone.

In [ ]:
# Pinned environment
!pip install -q --force-reinstall --no-deps "transformers==4.46.3"
!pip install -q pysbd camel-tools beautifulsoup4 lxml requests sentencepiece protobuf sacrebleu

In [ ]:
# Fixed seeds
import os, random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
GEN = torch.Generator(); GEN.manual_seed(SEED)      # passed to DataLoader so shuffling is fixed

# For bit-identical GPU results, uncomment the two lines below BEFORE any CUDA call
# They force deterministic kernels, but slow training and raise errors for ops that have no deterministic implementation.
# Seeds alone give reproducible convergence
# os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
# torch.use_deterministic_algorithms(True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| seed:', SEED)

In [ ]:
!pip install -q pysbd camel-tools beautifulsoup4 lxml requests
import re, requests, pysbd
from urllib.parse import urlsplit, urlunsplit, quote
from bs4 import BeautifulSoup

# CAMeL normalisation (real role), regex fallback
try:
    from camel_tools.utils.normalize import normalize_alef_ar, normalize_alef_maksura_ar, normalize_teh_marbuta_ar
    from camel_tools.utils.dediac import dediac_ar
    def normalize_ar(t): return normalize_teh_marbuta_ar(normalize_alef_maksura_ar(normalize_alef_ar(dediac_ar(t))))
except Exception:
    def normalize_ar(t):
        t=re.sub(r'[\u064B-\u0652\u0670\u0640]','',t); t=re.sub(r'[إأآا]','ا',t)
        return t.replace('ى','ي').replace('ة','ه')

# language routing
AR=re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]')
def arabic_ratio(t):
    L=[c for c in t if c.isalpha()]
    return sum(bool(AR.match(c)) for c in L)/len(L) if L else 0.0
def detect_lang(t): return 'ar' if arabic_ratio(t)>=0.4 else 'en'

# fetch
HEADERS={'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
         '(KHTML, like Gecko) Chrome/124.0 Safari/537.36 ProVe-Arabic-research/1.0',
         'Accept-Language':'ar,en;q=0.9',
         'Accept':'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'}
def _safe(u):
    p=urlsplit(u); return urlunsplit((p.scheme,p.netloc,quote(p.path),p.query,p.fragment))
def fetch_html(url):
    r=requests.get(_safe(url),headers=HEADERS,timeout=25)
    r.raise_for_status(); r.encoding=r.apparent_encoding or r.encoding
    return r.text

# clean
JUNK_TAGS=['script','style','noscript','nav','footer','header','aside','form','button','svg','iframe','figure','sup']
BLOCKS=['p','li','h1','h2','h3','h4','h5','h6','td','th','blockquote','caption','dd','dt']
LISTTOGGLE=re.compile(r'^\s*القائمة\s*\.{2,}\s*')
PUNCT_FIX=[(re.compile(r'\s+([.,؛،:!؟…\)\]])'),r'\1'),(re.compile(r'([(\[])\s+'),r'\1'),(re.compile(r'\s{2,}'),' ')]
def tidy_spacing(t):
    for p,r in PUNCT_FIX: t=p.sub(r,t)
    return t.strip()
def clean_to_text(html):
    soup=BeautifulSoup(html,'lxml')
    for t in soup(JUNK_TAGS): t.decompose()
    seen,chunks=set(),[]
    for b in soup.find_all(BLOCKS):
        txt=tidy_spacing(LISTTOGGLE.sub('',b.get_text(' ',strip=True)))
        if not txt or txt in seen: continue
        seen.add(txt)
        if txt[-1] not in '.!?؟…؛،:': txt+='.'
        chunks.append(txt)
    text='\n'.join(chunks)
    if len(text)<200: text=tidy_spacing(soup.get_text('\n',strip=True))
    return re.sub(r'\n{2,}','\n',text).strip()

# segment (pysbd + terminal merge)
TERMINALS='.؟?!…؛'
_seg={}
def _segmenter(lang):
    if lang not in _seg: _seg[lang]=pysbd.Segmenter(language=('ar' if lang=='ar' else 'en'),clean=False)
    return _seg[lang]
def segment(text,lang,min_chars=20):
    seg=_segmenter(lang); raw=[]
    for line in text.split('\n'):
        line=line.strip()
        if line: raw+=[s.strip() for s in seg.segment(line) if s.strip()]
    merged,buf=[],''
    for s in raw:
        buf=f'{buf} {s}'.strip() if buf else s
        if buf[-1] in TERMINALS: merged.append(buf); buf=''
    if buf: merged.append(buf)
    seen,out=set(),[]
    for s in merged:
        if s in seen or sum(c.isalpha() for c in s)<5 or len(s)<min_chars: continue
        seen.add(s); out.append(s)
    return out

print("Stage B module ready.")

## 2 — Segmenter comparison

Two corrections are applied for fairness.

Scraped text is passed through a spacing-tidy step,
since irregular spacing from HTML extraction confuses a trained neural tokeniser more than a
rule-based splitter.

Identical post-processing is applied to all three tools, so the base
splitter is the only variable. Comparing pysbd's post-processed output against the others' raw
output would be meaningless.

In [ ]:
!pip install -q stanza spacy
import time, statistics, torch

html = fetch_html('https://ar.wikipedia.org/wiki/نجيب_محفوظ')
text = clean_to_text(html)
USE_GPU = torch.cuda.is_available()

def stats(name, sents, secs):
    lens=[len(s) for s in sents] or [0]
    print(f'{name:16s} n={len(sents):5d}  mean={statistics.mean(lens):4.0f}c  '
          f'median={statistics.median(lens):4.0f}c  frags(<20c)={sum(l<20 for l in lens):4d}  '
          f'runons(>400c)={sum(l>400 for l in lens):3d}  {secs:5.1f}s')

res={}
# 1) pysbd + our merge (the current approach)
t=time.time(); res['pysbd+merge']=segment(text,'ar'); stats('pysbd+merge',res['pysbd+merge'],time.time()-t)

# 2) Stanza Arabic — neural tokenizer + sentence splitter (the slow, high-quality one)
import stanza
stanza.download('ar', verbose=False)
nlp_st=stanza.Pipeline('ar',processors='tokenize',verbose=False,use_gpu=USE_GPU)
t=time.time(); doc=nlp_st(text); res['stanza']=[s.text.strip() for s in doc.sentences if s.text.strip()]
stats('stanza',res['stanza'],time.time()-t)

# 3) spaCy Arabic — rule-based sentencizer (the light one)
import spacy
nlp_sp=spacy.blank('ar'); nlp_sp.add_pipe('sentencizer'); nlp_sp.max_length=2_000_000
t=time.time(); doc=nlp_sp(text); res['spacy']=[s.text.strip() for s in doc.sents if s.text.strip()]
stats('spacy',res['spacy'],time.time()-t)

# qualitative: how each renders the trilogy sentence
print('\n--- same passage, three tools ---')
for name,sents in res.items():
    hit=next((s for s in sents if 'كفاح طيبة' in s and 'رادوبيس' in s),
             next((s for s in sents if 'كفاح طيبة' in s),'(fragmented / not found)'))
    print(f'[{name}]\n  {hit[:280]}\n')

print('--- first 3 substantial (>80c) sentences per tool ---')
for name,sents in res.items():
    print(f'[{name}]')
    for s in [x for x in sents if len(x)>80][:3]: print('  •',s[:200])
    print()

### Shared post-processing

The same merge, deduplication, and minimum-length filtering applied to every splitter's output.

In [ ]:
# postprocess: same merge/dedup logic as segment()
# applied to each raw splitter's output for a fair comparison
import statistics
TERMINALS='.؟?!…؛'
def postprocess(sents,min_chars=20):
    merged,buf=[],''
    for s in sents:
        s=' '.join(s.split())
        if not s: continue
        buf=f'{buf} {s}'.strip() if buf else s
        if buf[-1] in TERMINALS: merged.append(buf); buf=''
    if buf: merged.append(buf)
    seen,out=set(),[]
    for s in merged:
        if s in seen or sum(c.isalpha() for c in s)<5 or len(s)<min_chars: continue
        seen.add(s); out.append(s)
    return out

### Results

In [ ]:
import statistics
ps=_segmenter('ar')
def clean(url): return clean_to_text(fetch_html(url))
def pysbd_raw(t):  return [x for x in ps.segment(t) if x.strip()]
def stanza_raw(t): return [s.text for s in nlp_st(t).sentences]
def spacy_raw(t):  return [s.text for s in nlp_sp(t).sents]
def show(label,segs,n=12,w=95):
    print(f'  [{label}] -> {len(segs)} segments')
    for i,s in enumerate(segs[:n],1): print(f'    {i:2d}. {" ".join(s.split())[:w]}')
    if len(segs)>n: print(f'    ... (+{len(segs)-n} more)')
    print()

# PART 1: MULTI-PAGE VALIDATION (identical post-processing for all)
PAGES={'Wiki: Mahfouz (prose+infobox)':'https://ar.wikipedia.org/wiki/نجيب_محفوظ',
       'Wiki: Egypt (table/list-heavy)':'https://ar.wikipedia.org/wiki/مصر',
       'Wiki: AI (technical prose)':'https://ar.wikipedia.org/wiki/ذكاء_اصطناعي',
       'UN.org Arabic (non-wiki)':'https://www.un.org/ar/about-us/universal-declaration-of-human-rights'}
def line(name,segs):
    L=[len(s) for s in segs] or [0]
    return (f'{name:7s} n={len(segs):5d} median={statistics.median(L):4.0f}c '
            f'frags(<20c)={sum(x<20 for x in L):4d} runons(>400c)={sum(x>400 for x in L):3d}')
print('============ MULTI-PAGE (post-processed) ============\n')
for title,url in PAGES.items():
    try: t=clean(url)
    except Exception as e: print(f'{title}\n  SKIPPED ({type(e).__name__})\n'); continue
    print(f'{title}  [{len(t)} chars]')
    for nm,fn in [('pysbd',pysbd_raw),('stanza',stanza_raw),('spacy',spacy_raw)]:
        print('  '+line(nm,postprocess(fn(t))))
    print()

# PART 2: CONCRETE SIDE-BY-SIDE
text=clean(PAGES['Wiki: Mahfouz (prose+infobox)'])
blocks=[b for b in text.split('\n') if b.strip()]
enders=lambda s:s.count('.')+s.count('؟')+s.count('!')+s.count('…')
para=next((b for b in blocks if enders(b)>=3 and 200<len(b)<700),
          next((b for b in blocks if enders(b)>=2 and len(b)>150),blocks[0]))

print('============ EXAMPLE A: one clean prose paragraph ============')
print('INPUT:\n ', ' '.join(para.split())[:600],'\n')
for nm,fn in [('pysbd',pysbd_raw),('stanza',stanza_raw),('spacy',spacy_raw)]: show(nm,fn(para))

print('============ EXAMPLE B: messy multi-block excerpt (real scraped shape) ============')
excerpt='\n'.join(blocks[blocks.index(para):blocks.index(para)+10])
print(f'INPUT: {len(excerpt)} chars across 10 blocks\n')
for nm,fn in [('pysbd',pysbd_raw),('stanza',stanza_raw),('spacy',spacy_raw)]: show(nm,fn(excerpt))